# Delivery time prediction – RandomForest
Trains a RandomForestRegressor on `dataset/train.csv` and saves the full pipeline with joblib.

In [ ]:
import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder

from features import CAT_COLS, NUM_COLS, build_features, parse_target

In [ ]:
df = pd.read_csv("dataset/train.csv")
y = parse_target(df["Time_taken(min)"])
X = df.drop(columns=["Time_taken(min)"])
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
X_train.shape, X_val.shape

In [ ]:
preprocess = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), NUM_COLS),
    ("cat", Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]), CAT_COLS),
])

model = Pipeline([
    ("features", FunctionTransformer(build_features)),
    ("preprocess", preprocess),
    ("rf", RandomForestRegressor(n_estimators=100, min_samples_leaf=10, n_jobs=-1, random_state=42)),
])
model.fit(X_train, y_train)

In [ ]:
pred = model.predict(X_val)
print("MAE :", mean_absolute_error(y_val, pred))
print("RMSE:", np.sqrt(mean_squared_error(y_val, pred)))
print("R2  :", r2_score(y_val, pred))

In [ ]:
# refit on all data, then save
model.fit(X, y)
joblib.dump(model, "model.joblib")

In [ ]:
# sanity check: reload and predict on test.csv
loaded = joblib.load("model.joblib")
test = pd.read_csv("dataset/test.csv")
loaded.predict(test)[:5]